# 03 · Silver con Change Data Feed

⏱ **15 min** · Slides: 9–12 (tablas Delta, MERGE, esquema) y 17–19 (Change Data Feed)

**Objetivo:** que silver se actualice **solo con lo que cambió** en bronze, usando el Change Data Feed (CDF) y `MERGE`.

```
bronze_schedule ──► table_changes(bronze) ──► aplanar ──► MERGE ──► silver_episodes
                    solo versiones nuevas                            (una fila por episodio)
```

| Paso | Qué haces |
|---|---|
| 1 | Ver los cambios de bronze con `table_changes` |
| 2 | Aplanar el JSON |
| 3 | Construir silver (CDF + MERGE) |
| 4 | Re-ejecutar: nada que hacer |
| 5 | Llega un día nuevo: solo entran esos episodios |
| 6 | Una corrección y un borrado en bronze llegan a silver |
| 7 | Ver los cambios de silver (los leerá gold) |

**Requiere:** notebook 02.

In [0]:
%run ./00_Setup

## Paso 1 · Ver los cambios de bronze

`table_changes('tabla', versión_inicial)` devuelve cada fila que cambió con tres columnas extra:
`_change_type`, `_commit_version` y `_commit_timestamp`.

In [0]:
%sql
SELECT _commit_version, _commit_timestamp, _change_type, count(*) AS filas
FROM table_changes('bronze_schedule', 0)
GROUP BY 1, 2, 3
ORDER BY _commit_version

In [0]:
%sql
SELECT id, name, source_file, _change_type, _commit_version
FROM table_changes('bronze_schedule', 0)
LIMIT 5

👀 **Qué observar:** hasta ahora todo es `insert` (las cargas de Auto Loader). Cada versión del log es un lote de cambios.

## Paso 2 · Aplanar el JSON

`to_silver()` (en `00_Setup`) convierte el JSON anidado en columnas con tipos correctos.

In [0]:
display(to_silver(spark.table(BRONZE)).limit(5))

## Paso 3 · Construir silver con CDF + MERGE

`build_silver()` hace cuatro cosas. Lee el código antes de ejecutarlo:

In [0]:
print(inspect.getsource(build_silver))

In [0]:
build_silver()

In [0]:
print("Filas en silver:", spark.table(SILVER).count())
display(spark.table(STATE))

👀 **Qué observar:** `pipeline_state` recuerda hasta qué versión de bronze se procesó. La próxima vez, silver
pedirá los cambios **desde la siguiente versión**.

## Paso 4 · Re-ejecutar: nada que hacer

In [0]:
build_silver()

## Paso 5 · Llega un día nuevo

RAW → bronze (Auto Loader) → silver (CDF).

In [0]:
download_days(next_day(), 1)
run_bronze()

In [0]:
desde = last_processed("silver") + 1
display(read_changes(BRONZE, desde, table_version(BRONZE))
        .groupBy("_commit_version", "_change_type").count())

In [0]:
build_silver()
print("Filas en silver:", spark.table(SILVER).count())

👀 **Qué observar:** silver procesó solo los episodios del día nuevo, no toda la tabla bronze.

## Paso 6 · Una corrección y un borrado en bronze

En la vida real bronze no se edita a mano: las correcciones llegan de la fuente. Aquí las **simulamos** para ver
cómo viajan por el CDF: un `UPDATE` produce dos filas (`update_preimage` y `update_postimage`) y un `DELETE`, una.

In [0]:
ep_update, ep_delete = [r["id"] for r in spark.table(BRONZE).select("id").orderBy("id").limit(2).collect()]

spark.sql(f"UPDATE {BRONZE} SET runtime = 99 WHERE id = {ep_update}")
spark.sql(f"DELETE FROM {BRONZE} WHERE id = {ep_delete}")
print("Actualizado:", ep_update, "| Borrado:", ep_delete)

In [0]:
desde = last_processed("silver") + 1
display(read_changes(BRONZE, desde, table_version(BRONZE))
        .select("id", "name", "runtime", "_change_type", "_commit_version")
        .orderBy("_commit_version", "id"))

In [0]:
build_silver()

print("runtime_min en silver del episodio actualizado:",
      spark.table(SILVER).where(F.col("episode_id") == ep_update).select("runtime_min").first()[0])
print("¿Sigue en silver el episodio borrado?:",
      spark.table(SILVER).where(F.col("episode_id") == ep_delete).count() > 0)

👀 **Qué observar:** la imagen previa no se aplica; la posterior actualiza la fila; el `delete` la elimina.
Silver procesó **2 cambios**, no miles de filas.

## Paso 7 · Los cambios de silver

Silver también tiene CDF activado: es lo que leerá gold en el siguiente notebook.

In [0]:
%sql
SELECT _commit_version, _change_type, count(*) AS filas
FROM table_changes('silver_episodes', 0)
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
show_history(SILVER)

👀 **Qué observar:** cada `build_silver()` dejó un `MERGE` en el historial, y el CDF de silver registra qué filas insertó, actualizó o borró.

## ✔ Checkpoint

- [ ] Sé leer `table_changes` y qué significa cada `_change_type`
- [ ] Silver se construyó con `MERGE` a partir de los cambios, no de toda la tabla
- [ ] `pipeline_state` guarda hasta qué versión se procesó
- [ ] Un `UPDATE` y un `DELETE` en bronze llegaron a silver

**Siguiente:** `04_Gold_con_CDF`